# Phase 0: reproduction of the starting point

Plan: `plans/phase_0.md` with its amendments. The numbers are computed by `analysis/phase0_reproduce.py` and `analysis/drift_reference.py`. The current run uses the corrected release `data/results/2026-09-23/` (outputs in `data/phase0/`); the first run used release `2026-09-13`, which had lost every `n't` (outputs in `data/phase0_release_2026-09-13/`). Both are shown.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
DATA = ROOT / "notebooks" / "roman" / "data"
NEW, OLD = DATA / "phase0", DATA / "phase0_release_2026-09-13"
RUNS = {"2026-09-23": NEW, "2026-09-13 (faulty)": OLD}
pd.set_option("display.max_colwidth", 120)

def both(name, index, cols):
    return pd.concat({k: pd.read_csv(d / name).set_index(index)[cols] for k, d in RUNS.items()}, axis=1)

## Environment

In [ ]:
pd.Series(json.loads((NEW / "environment.json").read_text()), name="value").to_frame()

CUDA is available on an RTX 4080, so the cross-encoder, embedding and NLI models of phase 1 run locally on the GPU. scikit-learn 1.6.1 with SciPy 1.18.1 emits a harmless warning about an unknown `iprint` solver option; the fitted models are unaffected.

## Year contrast classifier (`contrast.py`)

In [ ]:
both("contrast_auc.csv", ["view", "bank"], ["n", "auc", "null95", "p"])

Both releases give the recorded n, AUC and exact permutation p to three decimals; the restored contractions do not change the classifier. No view separates 2023 from 2024 better than the 95th percentile of the 70-split null.

## Non-answer phrase list (`qa_pairs.py`)

In [ ]:
both("nonanswer_by_bank.csv", "bank", ["pairs", "hits", "pairs_with_hit", "answer_words", "rate_per_10k"])

In [ ]:
pd.DataFrame({k: json.loads((d / "nonanswer_binomial.json").read_text()) for k, d in RUNS.items()})

On the corrected release the phrase list finds 24 hits instead of 19 (UBS 17, JPMorgan 7), and the rates rise to 5.2 and 1.7 per 10,000 answer words, with a binomial p of 0.013. The recorded 4.3 and 1.2 are reproduced only on the faulty release; the binomial test treats every word as independent and is anticonservative.

In [ ]:
new_hits = pd.read_csv(NEW / "nonanswer_hits.csv")
old_hits = pd.read_csv(OLD / "nonanswer_hits.csv")
key = ["bank", "quarter", "position_in_call", "phrase"]
added = new_hits.merge(old_hits[key], on=key, how="left", indicator=True).query("_merge == 'left_only'")
added[["bank", "quarter", "position_in_call", "phrase", "context"]]

The five added hits all contain a restored contraction. They have not been read against the manual labels of the brief; at least one (UBS 2023-Q4, “if we don’t get into a good place”) is a conditional clause rather than a refusal, so the recorded precision of 18 of 19 does not carry over automatically.

In [ ]:
new_hits[["bank", "quarter", "position_in_call", "phrase", "context"]]

The UBS hits are spread over six calls, and 10 of 17 fall in 2024-Q1 to 2024-Q3, so the rate still depends heavily on a few calls.

## Drift between calls (`analysis/drift_reference.py`)

In [ ]:
both("drift_reference.csv", ["view", "bank"], ["rho", "p", "rho_brief", "p_brief"])

Both releases reproduce all four recorded values (one TF-IDF document per quarter, Spearman rho between lag and similarity over 28 call pairs, 5000 permutations of the quarter order). For constant speakers similarity falls with lag in both banks, more steeply at JPMorgan; for the analyst Q&A the UBS trend is weaker and not significant at 0.05.

The JPMorgan constant-speaker p-value of 0 means that none of the 5000 permutations reached the observed rho, so p < 0.0002. The two earlier reconstructions (min_df = 3, no split by view) gave different values and are kept in `drift_reconstruction.csv` for reference only.

## Data handling tests

`tests/` checks sentence counts per call against the release README, uniqueness of the stable key, that pairs never cross calls, that questions are analyst turns and answers are management or IR turns, and that masking removes numbers and speaker names; since phase 1 it also checks question segments and answer alignment. All 16 tests pass on the corrected release. The name check covers earnings calls only, because the masking list in `contrast.py` is built from earnings-call speakers and misses at least one event-call speaker.